# Crypto Market Intelligence

This notebook builds the analysis end to end from the local DuckDB database and writes the four charts into `outputs/charts/`.

Run order:
1. `python src/download_binance_data.py`
2. `python src/load_to_duckdb.py`
3. Run all cells in this notebook.

The figures always reflect the period you downloaded, so they stay honest to the data.

In [ ]:
import sys
from pathlib import Path

# Make the src package importable when running from notebooks/
sys.path.append(str(Path.cwd().parent / 'src'))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from utils import connect, run_sql_file, run_query, save_chart

sns.set_theme(style='whitegrid')
pd.set_option('display.float_format', lambda v: f'{v:,.4f}')

## 1. Connect to the database

DuckDB runs in process, so there is no server to manage.

In [ ]:
con = connect(read_only=True)
con.execute('SELECT symbol, COUNT(*) AS rows, MIN(open_time) AS first, MAX(open_time) AS last FROM market_klines GROUP BY symbol').df()

## 2. Daily KPIs per asset

Realized volatility, volumes, trade counts and the taker buy ratio.

In [ ]:
kpis = run_sql_file(con, '02_market_kpis.sql')
kpis.head(10)

## 3. Volume anomalies

Hours where quote volume sits more than three standard deviations above the rolling 7 day baseline.

In [ ]:
anomalies = run_sql_file(con, '03_volume_anomalies.sql')
print(f'{len(anomalies)} anomaly hours flagged')
anomalies.head(10)

## 4. Volatility regimes

In [ ]:
regimes = run_sql_file(con, '04_volatility_regimes.sql')
regimes.groupby(['symbol', 'volatility_regime']).size().unstack(fill_value=0)

## 5. Session analysis

In [ ]:
sessions = run_sql_file(con, '05_session_analysis.sql')
sessions

## 6. Market health score

In [ ]:
health = run_sql_file(con, '06_market_health_score.sql')
health

## 7. Charts

Four figures, each saved to `outputs/charts/`.

### Chart 1: realized volatility by asset

In [ ]:
vol_by_asset = kpis.groupby('symbol')['realized_volatility'].mean().sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
vol_by_asset.plot(kind='barh', color='#F0B90B', ax=ax)
ax.set_title('Average realized volatility by asset')
ax.set_xlabel('Realized volatility (std of hourly log returns)')
ax.set_ylabel('')
save_chart(fig, 'volatility_by_asset.png')
plt.show()

### Chart 2: volume anomaly score

In [ ]:
top_anom = anomalies.sort_values('volume_z_score', ascending=False).head(15)
fig, ax = plt.subplots(figsize=(9, 5))
colors = top_anom['symbol'].astype('category').cat.codes
ax.scatter(top_anom['volume_z_score'], range(len(top_anom)), c=colors, cmap='autumn', s=80)
ax.set_yticks(range(len(top_anom)))
ax.set_yticklabels(top_anom['symbol'] + '  ' + top_anom['open_time'].astype(str).str[:13])
ax.set_title('Top volume anomalies by z-score')
ax.set_xlabel('Volume z-score (7 day rolling baseline)')
save_chart(fig, 'volume_anomaly_score.png')
plt.show()

### Chart 3: session volume heatmap

In [ ]:
heat = con.execute("""
    SELECT symbol, EXTRACT('hour' FROM open_time) AS utc_hour, AVG(quote_volume) AS avg_quote_volume
    FROM market_klines GROUP BY symbol, utc_hour ORDER BY symbol, utc_hour
""").df()
pivot = heat.pivot(index='symbol', columns='utc_hour', values='avg_quote_volume')
fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(pivot, cmap='YlOrBr', ax=ax, cbar_kws={'label': 'avg quote volume'})
ax.set_title('Average quote volume by UTC hour')
ax.set_xlabel('UTC hour')
ax.set_ylabel('')
save_chart(fig, 'session_volume_heatmap.png')
plt.show()

### Chart 4: market health score

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
h = health.sort_values('market_health_score')
ax.barh(h['symbol'], h['market_health_score'], color='#2EBD85')
ax.set_title('Market health score by asset (last 24h)')
ax.set_xlabel('Score (higher is calmer and more liquid)')
for i, v in enumerate(h['market_health_score']):
    ax.text(v + 1, i, f'{v:.0f}', va='center')
save_chart(fig, 'market_health_score.png')
plt.show()

In [ ]:
con.close()
print('Charts written to outputs/charts/')